In [11]:
#Imports
import pandas as pd
import plotly.express as px
import plotly.io as pio
import google.generativeai as genai
from google.genai import types
from google import genai
import csv
import os
import numpy as np
from plotly.subplots import make_subplots 
import plotly.graph_objects as go

In [12]:
#Helper Methods
def convert_to_numbers(x):
    if isinstance(x, str):
        x = x.strip().upper()
        if x.endswith("K"):
            return float(x[:-1]) * 1000
    return pd.to_numeric(x, errors='coerce')

In [7]:
csv_files = [
    '../Data/TimingData/2021-timing.csv',
    '../Data/TimingData/2022-timing.csv',
    '../Data/TimingData/2023-timing.csv',
    '../Data/TimingData/2024-timing.csv'
]

color_map = {
    'COMPLETED': 'green',
    'FAILED/NODE_FAIL': 'red',
    'TIMEOUT': 'yellow',
    'OUT_OF_MEMORY': 'orange',
    'RESIZING/REQUEUED': 'blue',
    'CANCELLED': 'black'
}

state_group_map = {
    'FAILED': 'FAILED/NODE_FAIL',
    'NODE_FAIL': 'FAILED/NODE_FAIL',
    'RESIZING': 'RESIZING/REQUEUED',
    'REQUEUED': 'RESIZING/REQUEUED',
    'CANCELLED': 'CANCELLED',
    'COMPLETED': 'COMPLETED',
    'TIMEOUT': 'TIMEOUT',
    'OUT_OF_MEMORY': 'OUT_OF_MEMORY'
}

all_states = set(color_map.keys())

combined_node_v_time = make_subplots(
    rows=4, cols=1,
    subplot_titles=[f"Node vs Wait Time - Actual Time Difference (s) for {csv_file.split('/')[-1][:4]}" for csv_file in csv_files],
    vertical_spacing=0.08
)

seen_states = set()

for idx, csv_file in enumerate(csv_files):
    df = pd.read_csv(csv_file)
    df['Nodes'] = df['Nodes'].apply(convert_to_numbers)

    states = df.iloc[:, 2]
    states = states.str.replace(r'^CANCELLED.*', 'CANCELLED', regex=True)
    df['clean_state'] = states.map(state_group_map)

    backfilled = df.iloc[:, 3].astype(str).str.lower()
    df['backfilled_label'] = backfilled.apply(lambda x: 'yes' if x == 'yes' else 'no')

    year = csv_file.split('/')[-1][:4]

    px_fig = px.scatter(
        df,
        x='Nodes',
        y='Time Difference (s)',
        color='clean_state',
        symbol='backfilled_label',
        color_discrete_map=color_map,
        symbol_sequence=['circle', 'cross']
    )

    if year in ['2021', '2024']:
        combined_node_v_time.update_yaxes(range=[0, 50000], row=idx + 1, col=1)

    for trace in px_fig.data:
        state, backfilled = [part.strip() for part in trace.name.split(',')]
        key = (state, backfilled)

        backfilled_text = '(Backfilled)' if backfilled == 'yes' else ''
        trace.name = f"{state} {backfilled_text}"

        if key in seen_states:
            trace.showlegend = False
        else:
            trace.showlegend = True
            seen_states.add(key)

        combined_node_v_time.add_trace(trace, row=idx + 1, col=1)
        combined_node_v_time.update_xaxes(title_text='Number of Nodes', row=idx + 1, col=1)
        combined_node_v_time.update_yaxes(title_text='Time Difference (s)', row=idx + 1, col=1)

for i in range(4):
    combined_node_v_time.update_xaxes(tickformat='~s', row=i + 1, col=1)

combined_node_v_time.update_layout(
    height=1600,
    width=1200,
    title_text="Node vs Expected - Actual Runtime(s) Across Years",
    legend_title_text='Job State',
    showlegend=True,
)

combined_node_v_time.write_html('../Plots/TimingPlots/Combined_Node_vs_Diffsec_AllYears.html')
#combined_node_v_time.show()



In [8]:
#Individual html plots for Node_vs_Diffsec plots

for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    df['Nodes'] = df['Nodes'].apply(convert_to_numbers)

    states = df.iloc[:, 2]
    states = states.str.replace(r'^CANCELLED.*', 'CANCELLED', regex=True)
    df['clean_state'] = states.map(state_group_map)

    backfilled = df.iloc[:, 3].astype(str).str.lower()
    df['backfilled_label'] = backfilled.apply(lambda x: 'yes' if x == 'yes' else 'no')

    year = csv_file.split('/')[-1][:4]

    node_v_time = px.scatter(
        df,
        x='Nodes',
        y='Time Difference (s)',
        color='clean_state',
        symbol='backfilled_label',
        color_discrete_map=color_map,
        symbol_sequence=['circle', 'cross'],
        title=f"Node vs Expected - Actual Runtime(s) for {year}",
        labels={
            'Nodes': 'Number of Nodes',
            'Diffsec': 'Difference in Seconds',
            'clean_state': 'Job State'
        }
    )

    for trace in node_v_time.data:
        # trace.name is like "COMPLETED, yes" or "TIMEOUT, no"
        state, backfilled = [s.strip() for s in trace.name.split(',')]
        backfilled_text = " (Backfilled)" if backfilled == "yes" else ""
        trace.name = f"{state}{backfilled_text}"

    if year in ['2021', '2024']:
        node_v_time.update_yaxes(range=[-100, 50000])
        node_v_time.update_xaxes(range=[-100, 10000])
    elif year in ['2022']:
        node_v_time.update_yaxes(range=[-100, 650000])
        node_v_time.update_xaxes(range=[-100, 10000])
    elif year in ['2023']:
        node_v_time.update_xaxes(range=[-100, 10000])
        node_v_time.update_yaxes(range=[-100, 100000])

    node_v_time.update_layout(
        height=1000,
        width=1500,
        legend_title_text='Job State',
        showlegend=True
    )

    node_v_time.write_html(f'../Plots/TimingPlots/Node_vs_Diffsec_{year}.html')
    #node_v_time.show()

# Task 11

In [15]:
#Part A
#number of jobs over time
num_jobs_time = [[258618, 3455591], 
                [98122, 7366544], 
                [883194, 6288328], 
                [355641, 5115144]]

jobs_over_time = pd.DataFrame(num_jobs_time, columns=['Jobs', 'Jobsteps'], index=['2021', '2022', '2023', '2024'])

jobs_time_plot = go.Figure()
jobs_time_plot.add_trace(go.Bar(x=jobs_over_time.index, y=jobs_over_time['Jobs'], name='Jobs'))
jobs_time_plot.add_trace(go.Bar(x=jobs_over_time.index, y=jobs_over_time['Jobsteps'], name='Jobsteps'))

jobs_time_plot.update_layout(
    barmode='group',
    title='Number of Jobs Over Time',
    xaxis_title="Year",
    yaxis_title='Number of Jobs',
    legend_title='Type',
)

jobs_time_plot.show()
jobs_time_plot.write_image('../Plots/Num_Jobs_Over_Time.png')
jobs_time_plot.write_html('../Plots/Num_Jobs_Over_Time.html')

